In [1]:
endpoint_name = 'math-proficiency-endpoint'

In [2]:
#Look at recent scaling activity

import boto3
from datetime import datetime, timedelta

def get_endpoint_info(endpoint_name):
    sagemaker_client = boto3.client('sagemaker')
    autoscaling_client = boto3.client('application-autoscaling')
    cloudwatch_client = boto3.client('cloudwatch')

    # Get current instance count
    response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    current_count = response['ProductionVariants'][0]['CurrentInstanceCount']
    print(f"Current Instance Count: {current_count}")

    # Get auto-scaling activities
    scalable_target_id = f"endpoint/{endpoint_name}/variant/AllTraffic"
    scaling_activities = autoscaling_client.describe_scaling_activities(
        ServiceNamespace='sagemaker',
        ResourceId=scalable_target_id,
        MaxResults=10  # Adjust as needed
    )
    
    print("\nRecent Scaling Activities:")
    for activity in scaling_activities['ScalingActivities']:
        print(f"Time: {activity['StartTime']}, Description: {activity['Description']}")

    # Get instance count metric data
    end_time = datetime.utcnow()
    start_time = end_time - timedelta(hours=24)  # Last 24 hours
    
    metric_data = cloudwatch_client.get_metric_data(
        MetricDataQueries=[
            {
                'Id': 'm1',
                'MetricStat': {
                    'Metric': {
                        'Namespace': 'AWS/ApplicationAutoScaling',
                        'MetricName': 'GroupDesiredCapacity',
                        'Dimensions': [
                            {'Name': 'ServiceNamespace', 'Value': 'sagemaker'},
                            {'Name': 'ResourceId', 'Value': scalable_target_id}
                        ]
                    },
                    'Period': 300,
                    'Stat': 'Average'
                }
            }
        ],
        StartTime=start_time,
        EndTime=end_time
    )
    
    print("\nInstance Count Over Time:")
    for timestamp, value in zip(metric_data['MetricDataResults'][0]['Timestamps'], 
                                metric_data['MetricDataResults'][0]['Values']):
        print(f"Time: {timestamp}, Desired Capacity: {value}")

# Usage
get_endpoint_info(endpoint_name)

Current Instance Count: 1

Recent Scaling Activities:

Instance Count Over Time:
